# 08. Count objects and signal a zone alarm on LED0

Use a **USB camera connected to the PYNQ board** (for example Logitech C270), not the laptop webcam. This notebook reuses the installed Tinier-YOLO model from [05](05_tiny-yolo-image.ipynb): 20 VOC classes including person, car, bicycle, dog, bottle and chair.

![Tinier-YOLO topology](Tinier-YOLO-topology.svg)

Run cells in order with **Shift + Enter**. Shut down other FPGA/camera notebook kernels first. The installed `qnn` package must already have its unused `Xlnk` imports removed. No new model, bitstream compilation, package reinstall or training is required.

Each button press captures a **new still image**, closes the camera, and processes that image. There is no background camera thread. CPU convolutions can take time; this is not a smooth live-video demonstration. Previous results stay visible. Captures and results are not saved as separate files.

A rectangle marks the target zone. An alarm is raised when the **center of a detected target box** lies inside it. Counts describe objects in the **current snapshot**; this does not track people or count unique visitors across captures.

**Storage:** no result folders, PNGs, scores or JSON logs are saved beside this notebook. The legacy native library uses small temporary files in `/dev/shm` (RAM), removed after processing and on session cleanup. History stores reduced previews in memory; restart the kernel to clear it. Jupyter can still embed displayed images when saving the notebook itself; use **Cell → All Output → Clear**, then save, to remove them before the next group. Original example images and model files are unchanged.

## 1. Check packages, model and camera settings

Run this cell. It locates the existing model and custom Darknet installation and lists `/dev/video*`. The default camera is `/dev/video0`, requesting 640×480. If your camera uses another device, change `CAMERA_DEVICE` after checking the list. The returned frame dimensions are used even if the camera chooses another resolution.

Missing files are reported before FPGA initialization. `ipywidgets` provides the buttons when available; the notebook also supports manual function calls. The native OpenCV/Darknet packages are those already used on the board.

In [ ]:
import sys
import os
import json
import io
import traceback
from collections import Counter
from contextlib import nullcontext
import cv2
import gc
import random
import time
import ctypes
import tempfile
import os
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from matplotlib import pyplot as plt
from IPython.display import display
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

import qnn
from qnn import TinierYolo, utils

QNN_PARAM_DIR = Path(qnn.__file__).resolve().parent / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "Tinier-YOLO-topology.svg").is_file():
    NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
IMAGE_DIR = NOTEBOOK_DIR / "yoloimages"
DARKNET_DIR = Path("/opt/darknet")
VOC_NAMES = DARKNET_DIR / "data/voc.names"
required_files = [QNN_PARAM_DIR / name for name in (
    "tinier-yolo-layers.json", "tinier-yolo-conv0-W.npy", "tinier-yolo-conv8-W.npy",
    "tinier-yolo-conv0-bias.npy", "tinier-yolo-conv8-bias.npy",
    "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg")]
required_files += [DARKNET_DIR / "python/darknet.py", DARKNET_DIR / "libdarknet.so", VOC_NAMES]
# Darknet loads all printable-character PNGs at each size when drawing labels.
required_files += [DARKNET_DIR / "data/labels" / "{}_{}.png".format(char, size)
                   for size in range(8) for char in range(32, 127)]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing (first 12):\n" + "\n".join(missing[:12]))
if str(DARKNET_DIR / "python") not in sys.path:
    sys.path.insert(0, str(DARKNET_DIR / "python"))
import darknet as dn

dn.lib.parse_network_cfg.argtypes = [ctypes.c_char_p]
dn.lib.parse_network_cfg.restype = ctypes.c_void_p
dn.lib.forward_region_layer_pointer_nolayer.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_float)]
dn.lib.forward_region_layer_pointer_nolayer.restype = None
dn.lib.draw_detection_python.argtypes = [ctypes.c_void_p, ctypes.c_char_p,
    ctypes.c_float, ctypes.c_float, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p]
dn.lib.draw_detection_python.restype = None
dn.lib.free_network.argtypes = [ctypes.c_void_p]
dn.lib.free_network.restype = None
print("Parameters:", QNN_PARAM_DIR)
print("USB devices:", ", ".join(str(p) for p in sorted(Path("/dev").glob("video*"))) or "none")

CAMERA_DEVICE = "/dev/video0"
CAMERA_WIDTH, CAMERA_HEIGHT = 640, 480
WARMUP_FRAMES = 12
THRESHOLD = 0.3
CAPTURE_INDEX = 0
# Close controls from an earlier run before replacing model state.
for control in globals().get("ui_buttons", []):
    control.disabled = True
if globals().get("ui_panel") is not None:
    ui_panel.close()
print("Widgets:", "available" if widgets is not None else "not installed; use the manual call below")

def memory_preview(value):
    if isinstance(value, Image.Image):
        image = value.copy()
    else:
        with Image.open(str(value)) as opened:
            image = opened.convert("RGB")
    image.thumbnail((480, 360))
    return image

def clear_intermediate_files():
    directory = globals().get("RESULT_DIR")
    if directory is not None and directory.is_dir():
        for path in directory.iterdir():
            if path.is_file():
                path.unlink()



## 2. Load the model once

Wait for **Model ready**. The first and last convolutions use the ARM CPU, intermediate quantized layers use the FPGA, and Darknet handles letterboxing. Large weights are not downloaded again. Rerunning this section starts a new in-memory history; close the previous controls first with **Finish / release**.

In [ ]:
def release_model():
    global classifier, net_darknet
    try:
        if globals().get("classifier") is not None:
            classifier.deinit_accelerator()
    finally:
        classifier = None
        if globals().get("net_darknet"):
            dn.lib.free_network(net_darknet)
        net_darknet = None
        if globals().get("_result_workspace") is not None:
            _result_workspace.cleanup()

release_model()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Loading weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv8_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv0_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
conv8_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
classifier = None
net_darknet = None
if globals().get("_result_workspace") is not None:
    _result_workspace.cleanup()
if not Path("/dev/shm").is_dir():
    raise RuntimeError("RAM-backed /dev/shm is required; no SD-card fallback is used.")
_result_workspace = tempfile.TemporaryDirectory(prefix="qnn-08-zone-", dir="/dev/shm")
RESULT_DIR = Path(_result_workspace.name)
try:
    classifier = TinierYolo()
    net = classifier.load_network()
    out_ch, out_dim, _ = net["conv7"]["output"]
    first_ch, first_dim, _ = net["conv1"]["input"]
    if conv0_weights.shape[0] != first_ch or conv0_bias.shape != (first_ch, 1):
        raise ValueError("First-convolution weights do not match the network.")
    if conv8_weights.shape[:2] != (125, out_ch) or conv8_bias.shape != (125, 1):
        raise ValueError("Expected the supplied 20-class, 5-anchor VOC detection head.")
    conv0_bias = np.broadcast_to(conv0_bias, (first_ch, first_dim * first_dim))
    conv8_bias = np.broadcast_to(conv8_bias, (125, out_dim * out_dim))
    net_darknet = dn.lib.parse_network_cfg(str(
        QNN_PARAM_DIR / "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg").encode())
    if not net_darknet:
        raise RuntimeError("Darknet failed to load its network configuration.")
    print("Initializing FPGA...", flush=True)
    classifier.init_accelerator()
except BaseException:
    release_model()
    raise
# Only temporary native-library files are placed in RAM.
RESULT_INDEX = 0
detection_history = []
print("Model ready. Intermediate files use RAM and are removed after processing.")

CAPTURE_INDEX = 0


## 3. Define model and camera helpers

Run this cell. It defines the existing model stages and safe camera capture. Each capture warms up the USB camera and releases it in `finally`, before slow inference begins. A temporary PNG in RAM is used by the legacy native library and read by the same model preprocessing as notebook 05. This cell itself does not open the camera.

In [ ]:
def load_detector_image(image_path):
    original = None
    letterboxed = None
    try:
        original = dn.load_image(str(image_path).encode(), 0, 0)
        if not original.data:
            raise RuntimeError("Darknet could not read " + str(image_path))
        letterboxed = dn.letterbox_image(original, 416, 416)
        if not letterboxed.data:
            raise RuntimeError("Darknet letterboxing failed.")
        pixels = np.copy(np.ctypeslib.as_array(letterboxed.data, (3, 416, 416)))
        # Preserve the original trained model's paired spatial-axis swaps.
        return np.swapaxes(pixels, 0, 2)[np.newaxis, :, :, :]
    finally:
        if letterboxed is not None and letterboxed.data:
            dn.free_image(letterboxed)
        if original is not None and original.data:
            dn.free_image(original)

def first_convolution(pixels):
    if pixels.shape != (1, 416, 416, 3):
        raise ValueError("Expected one letterboxed 416x416 RGB image.")
    values = utils.conv_layer(pixels, conv0_weights, b=conv0_bias, stride=2, padding=1)
    return utils.quantize(values.clip(0.0, 4.0) / 4.0, 3)

def fpga_layers(quantized):
    if classifier is None:
        raise RuntimeError("Model released. Rerun section 2 before inference.")
    output = classifier.get_accel_buffer(out_ch, out_dim)
    inputs = np.ascontiguousarray(classifier.prepare_buffer(quantized * 7))
    started = time.perf_counter()
    classifier.inference(inputs, output)
    call_seconds = time.perf_counter() - started
    values = classifier.postprocess_buffer(output)
    if values.size != out_ch * out_dim * out_dim or not np.all(np.isfinite(values)):
        raise ValueError("Invalid FPGA output.")
    return values, call_seconds

def last_convolution(values):
    pixels = np.swapaxes(values.reshape(out_dim, out_dim, out_ch), 0, 1)[np.newaxis]
    output = utils.conv_layer(pixels, conv8_weights, b=conv8_bias, stride=1)
    # The native region layer reads float*, so float32 and contiguous storage are required.
    output = np.ascontiguousarray(output, dtype=np.float32)
    if output.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(output)):
        raise ValueError("Invalid detection-head output.")
    return output

def draw_detections(image_path, values, threshold=0.3):
    global RESULT_INDEX
    if net_darknet is None:
        raise RuntimeError("Model released. Rerun section 2.")
    if not 0 < threshold < 1:
        raise ValueError("Detection threshold must be between 0 and 1.")
    # Defensive check also protects callers that rerun this cell independently.
    if values.dtype != np.float32 or not values.flags.c_contiguous:
        raise ValueError("Darknet input must be contiguous float32.")
    if values.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(values)):
        raise ValueError("Wrong detection-head shape or non-finite values.")
    RESULT_INDEX += 1
    result_name = "detection_{:04d}".format(RESULT_INDEX)
    output_prefix = RESULT_DIR / result_name
    output_image = RESULT_DIR / (result_name + ".png")
    probabilities = RESULT_DIR / (result_name + "_probabilities.txt")
    # This legacy C helper uses 256-byte path buffers, including added suffixes.
    paths = [str(image_path), str(VOC_NAMES), str(DARKNET_DIR) + "/",
             str(output_prefix), str(probabilities)]
    if any(len(path.encode()) > 240 for path in paths):
        raise ValueError("Use a shorter notebook/image path for the legacy Darknet helper.")
    for path in (output_image, probabilities):
        if path.exists():
            path.unlink()
    try:
        dn.lib.forward_region_layer_pointer_nolayer(
            net_darknet, values.ctypes.data_as(ctypes.POINTER(ctypes.c_float)))
        dn.lib.draw_detection_python(net_darknet, paths[0].encode(), ctypes.c_float(threshold),
            ctypes.c_float(0.5), paths[1].encode(), paths[2].encode(),
            paths[3].encode(), paths[4].encode())
        if not output_image.is_file() or not probabilities.is_file():
            raise RuntimeError("Darknet did not create the detection outputs.")
        detections = []
        for line in probabilities.read_text().splitlines():
            if not line.strip():
                continue
            name, score = line.rsplit(":", 1)
            detections.append((name.strip(), float(score.strip().rstrip("%")) / 100.0))
        return memory_preview(output_image), sorted(detections, key=lambda item: item[1], reverse=True)
    finally:
        for path in (output_image, probabilities):
            if path.exists():
                path.unlink()


def show_detections(output_image, detections):
    for name, score in detections:
        print("{:20s} {:6.0%}".format(name, score))
    if not detections:
        print("No objects above the selected threshold. This is a valid result.")
    preview = memory_preview(output_image)
    preview.thumbnail((800, 600))
    display(preview)

def capture_frame():
    """Open a board-connected USB camera, get a fresh frame, always close it."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError("Cannot open {}. Connect the USB camera to the board and close other camera kernels.".format(CAMERA_DEVICE))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
        for _ in range(WARMUP_FRAMES):
            ok, frame = cap.read()
            if not ok or frame is None or frame.size == 0:
                raise RuntimeError("No camera frame. Check USB power/connection and camera ownership.")
        return frame.copy()  # OpenCV BGR; cv2.imwrite preserves the correct PNG colors.
    finally:
        cap.release()

def save_capture(frame):
    global CAPTURE_INDEX
    CAPTURE_INDEX += 1
    path = RESULT_DIR / "capture_{:04d}.png".format(CAPTURE_INDEX)
    if not cv2.imwrite(str(path), frame):
        raise IOError("Could not save camera capture: " + str(path))
    return path

def preview_camera():
    frame = capture_frame()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    preview = Image.fromarray(rgb)
    preview.thumbnail((640, 480))
    print("Camera preview — captured now; the camera has been released.")
    display(preview)
    return frame

def run_model(image_path):
    if classifier is None:
        raise RuntimeError("Session finished. Rerun section 2 and the controls cell.")
    if not classifier.init:
        print("Loading YOLO overlay...", flush=True)
        classifier.init_accelerator()
    timings = {}
    started = time.perf_counter()
    pixels = load_detector_image(image_path)
    timings["letterbox_s"] = time.perf_counter() - started
    started = time.perf_counter()
    quantized = first_convolution(pixels)
    timings["first_cpu_s"] = time.perf_counter() - started
    started = time.perf_counter()
    fpga_output, timings["fpga_call_s"] = fpga_layers(quantized)
    timings["fpga_with_buffers_s"] = time.perf_counter() - started
    started = time.perf_counter()
    head = last_convolution(fpga_output)
    timings["last_cpu_s"] = time.perf_counter() - started
    return head, timings

def show_pair(original_path, result_path):
    original = memory_preview(original_path)
    annotated = memory_preview(result_path)
    original.thumbnail((480, 360))
    annotated.thumbnail((480, 360))
    canvas = Image.new("RGB", (original.width + annotated.width, max(original.height, annotated.height) + 28), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 7), "Captured frame", fill="black")
    draw.text((original.width + 8, 7), "Detection result", fill="black")
    canvas.paste(original, (0, 28))
    canvas.paste(annotated, (original.width, 28))
    display(canvas)


## 4. Decode boxes and define the counting rule

The original Darknet probability text has class names/scores but no coordinates. This cell defines a small Python decoder using the installed model's anchors, sigmoid/objectness, class softmax, letterbox correction and per-class non-maximum suppression (NMS). These computations run on the CPU; the neural-network stages are unchanged.

For counts, one highest-scoring label is retained per identical predicted box. NMS removes overlapping candidates of the same class, but false positives/duplicates can still occur. The colored dot marks the predicted center used by the zone rule; mere overlap with the zone does not trigger the alarm.

In [ ]:
# This decoder follows the supplied Darknet region-layer equations.
# Unlike its text output, these records include coordinates needed for ROI tests.
region_text = (QNN_PARAM_DIR / "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg").read_text().split("[region]", 1)[1]
region_settings = {}
for line in region_text.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        region_settings[key.strip()] = value.strip()
ANCHORS = np.array([float(x.strip()) for x in region_settings["anchors"].split(",")]).reshape(-1, 2)
if (region_settings.get("classes") != "20" or region_settings.get("num") != "5"
        or region_settings.get("coords") != "4" or region_settings.get("softmax") != "1"):
    raise ValueError("Expected the supplied five-anchor, 20-class softmax region head.")
VOC_CLASSES = VOC_NAMES.read_text().splitlines()
if len(VOC_CLASSES) != 20 or ANCHORS.shape != (5, 2):
    raise ValueError("VOC names/anchors do not match the installed model.")

def sigmoid(values):
    values = np.asarray(values, dtype=np.float64)
    return np.exp(-np.logaddexp(0.0, -values))

def box_iou(a, b):
    left, top = max(a[0], b[0]), max(a[1], b[1])
    right, bottom = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0.0, right - left) * max(0.0, bottom - top)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return intersection / max(area_a + area_b - intersection, 1e-12)

def decode_boxes(head, width, height, threshold=0.3, nms_threshold=0.3):
    if head.shape != (125, 13, 13) or not np.all(np.isfinite(head)):
        raise ValueError("Expected a finite 125x13x13 detection head.")
    if width <= 0 or height <= 0 or not 0 < threshold < 1 or not 0 <= nms_threshold <= 1:
        raise ValueError("Invalid image dimensions or detection/NMS threshold.")
    values = head.astype(np.float64).reshape(5, 25, 13, 13)
    xy = sigmoid(values[:, :2])
    objectness = sigmoid(values[:, 4])
    logits = values[:, 5:]
    exponentials = np.exp(logits - logits.max(axis=1, keepdims=True))
    scores = objectness[:, None] * exponentials / exponentials.sum(axis=1, keepdims=True)
    # Darknet letterbox_image and correct_region_boxes use integer resized dimensions.
    if 416 / width < 416 / height:
        new_width, new_height = 416, height * 416 // width
    else:
        new_width, new_height = width * 416 // height, 416
    if new_width < 1 or new_height < 1:
        raise ValueError("Image aspect ratio is too extreme.")
    candidates = []
    for anchor, row, col in np.argwhere(scores.max(axis=1) > threshold):
        cx = (col + xy[anchor, 0, row, col]) / 13
        cy = (row + xy[anchor, 1, row, col]) / 13
        # Very large logits yield boxes outside the frame; cap exponentiation safely.
        bw, bh = np.exp(np.clip(values[anchor, 2:4, row, col], -50, 50)) * ANCHORS[anchor] / 13
        cx = (cx - (416 - new_width) / 832) * 416 / new_width * width
        cy = (cy - (416 - new_height) / 832) * 416 / new_height * height
        bw, bh = bw * 416 / new_width * width, bh * 416 / new_height * height
        box = [float(cx-bw/2), float(cy-bh/2), float(cx+bw/2), float(cy+bh/2)]
        for class_id in np.flatnonzero(scores[anchor, :, row, col] > threshold):
            candidates.append(dict(class_id=int(class_id), label=VOC_CLASSES[class_id],
                                   score=float(scores[anchor, class_id, row, col]), box=box))
    # Darknet uses per-class NMS; keep unclipped boxes for the same IoU geometry.
    kept = []
    for candidate in sorted(candidates, key=lambda d: d["score"], reverse=True):
        if not any(candidate["class_id"] == previous["class_id"] and
                   box_iou(candidate["box"], previous["box"]) > nms_threshold for previous in kept):
            kept.append(candidate)
    # For counting, retain one highest-scoring label per identical anchor box.
    detections, seen_boxes = [], set()
    for candidate in kept:
        key = tuple(candidate["box"])
        if key in seen_boxes:
            continue
        seen_boxes.add(key)
        x1, y1, x2, y2 = candidate["box"]
        clipped = [max(0.0, min(width-1.0, x1)), max(0.0, min(height-1.0, y1)),
                   max(0.0, min(width-1.0, x2)), max(0.0, min(height-1.0, y2))]
        if clipped[2] <= clipped[0] or clipped[3] <= clipped[1]:
            continue
        item = dict(candidate)
        item["box"] = clipped
        # Test the un-clipped predicted center, rather than shifting it at image edges.
        item["center"] = [(x1+x2)/2, (y1+y2)/2]
        detections.append(item)
    return detections

def apply_zone(detections, width, height, roi, target):
    x1, y1, x2, y2 = roi
    if not 0 <= x1 < x2 <= 1 or not 0 <= y1 < y2 <= 1:
        raise ValueError("ROI must satisfy 0 <= left < right <= 1 and 0 <= top < bottom <= 1.")
    if target not in VOC_CLASSES:
        raise ValueError("Choose a target from VOC_CLASSES.")
    counts = Counter(d["label"] for d in detections)
    inside = []
    for detection in detections:
        cx, cy = detection["center"]
        if detection["label"] == target and x1*width <= cx <= x2*width and y1*height <= cy <= y2*height:
            inside.append(detection)
    return dict(counts=dict(counts), target=target, inside=inside, alarm=bool(inside))

def annotate_zone(image_path, detections, roi, decision):
    with Image.open(str(image_path)) as opened:
        image = opened.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    roi_box = tuple(int(v * dim) for v, dim in zip(roi, (width, height, width, height)))
    alarm_color = "red" if decision["alarm"] else "green"
    draw.rectangle(roi_box, outline=alarm_color, width=3)
    draw.text((roi_box[0]+5, max(0, roi_box[1]-16)), "TARGET ZONE", fill=alarm_color)
    for detection in detections:
        color = "red" if any(detection is item for item in decision["inside"]) else "yellow"
        box = tuple(int(v) for v in detection["box"])
        draw.rectangle(box, outline=color, width=2)
        cx, cy = detection["center"]
        draw.ellipse((int(cx)-3, int(cy)-3, int(cx)+3, int(cy)+3), fill=color)
        draw.text((box[0]+3, max(0, box[1]-13)), "{} {:.0%}".format(detection["label"], detection["score"]), fill=color)
    banner = Image.new("RGB", (width, height+32), "white")
    banner.paste(image, (0, 32))
    ImageDraw.Draw(banner).text((6, 10), "{} | {} in zone: {}".format(
        "ALARM" if decision["alarm"] else "CLEAR", decision["target"], len(decision["inside"])), fill=alarm_color)
    return banner


## 5. Configure the zone, target class and physical LED

Defaults: target **person**, central zone, **LED0 enabled**. `ROI = (left, top, right, bottom)` uses fractions from 0 to 1 measured from the top-left of the **whole captured frame**. For example, `(0.0, 0.0, 0.5, 1.0)` selects the left half. Adjust the tuple here and rerun this cell, or assign `ROI`/`TARGET_CLASS` in a new cell before the next capture.

The YOLO overlay has **no LED GPIO**. For a physical alarm, each completed inference releases the accelerator and loads the installed **base.bit** overlay to set LED0. The next detection reloads YOLO automatically. This adds loading time; the LED represents the **last processed snapshot**, not continuous monitoring, and its indication is not maintained while YOLO is loaded. Do not load another overlay yourself during inference.

Physical LED mode targets **PYNQ-Z1/Z2**. Set `PHYSICAL_LED = False` for a screen-only alarm, or if base.bit/board LED control is unavailable. No extra wiring is needed. Reconfiguring this cell switches off an earlier alarm first.

In [ ]:
if "turn_led_off" in globals():
    turn_led_off()
# The YOLO bitstream has no LED AXI GPIO. Reuse base.bit only after deinitializing YOLO.
PHYSICAL_LED = True
LED_INDEX = 0
ROI = (0.25, 0.20, 0.75, 0.90)  # left, top, right, bottom; fractions of the whole frame.
TARGET_CLASS = "person"
NMS_THRESHOLD = 0.3
base_overlay = None

def set_alarm_led(active):
    global base_overlay
    if not PHYSICAL_LED:
        print("Physical LED disabled; screen alarm:", "ON" if active else "OFF")
        return
    if os.environ.get("BOARD") not in ("Pynq-Z1", "Pynq-Z2"):
        raise RuntimeError("Physical LED mode is configured for Pynq-Z1/Z2. Set PHYSICAL_LED=False on another board.")
    # Release accelerator buffers BEFORE replacing the FPGA configuration.
    classifier.deinit_accelerator()
    from pynq.overlays.base import BaseOverlay
    if base_overlay is None or not base_overlay.is_loaded():
        base_overlay = BaseOverlay("base.bit")
    if not 0 <= LED_INDEX < len(base_overlay.leds):
        raise ValueError("LED_INDEX is not available in the base overlay.")
    base_overlay.leds[LED_INDEX].write(int(bool(active)))

def turn_led_off():
    if base_overlay is not None and base_overlay.is_loaded():
        base_overlay.leds[LED_INDEX].off()


## 6. Define capture, counting and alarm actions

No capture starts yet. The action prints class counts and the number of target detections inside the zone, draws the zone green for **CLEAR** or red for **ALARM**, and updates LED0 when enabled. Results include individual boxes/centers, zone settings, counts, alarm state and timings in memory.

The LED is reset before a new inference and switched off on cleanup/error. If LED control fails, the successful image result is still retained and the error is printed explicitly.

In [ ]:
def capture_and_detect():
    if classifier is None:
        raise RuntimeError("Session finished. Rerun section 2, then the controls cell.")
    # Validate before accessing the camera or changing an existing LED indication.
    apply_zone([], 640, 480, ROI, TARGET_CLASS)
    if not 0 < THRESHOLD < 1 or not 0 <= NMS_THRESHOLD <= 1:
        raise ValueError("Invalid detection or NMS threshold.")
    if PHYSICAL_LED and os.environ.get("BOARD") not in ("Pynq-Z1", "Pynq-Z2"):
        raise RuntimeError("Set PHYSICAL_LED=False on this board.")
    started = time.perf_counter()
    print("Capturing USB camera...", flush=True)
    try:
        frame = capture_frame()
        capture_path = save_capture(frame)
        print("Captured", capture_path.name, "— processing; please wait...", flush=True)
        # The previous alarm applies only to the previous snapshot. Clear it before a new inference.
        turn_led_off()
        head, timings = run_model(capture_path)
        height, width = frame.shape[:2]
        post_started = time.perf_counter()
        detections = decode_boxes(head, width, height, THRESHOLD, NMS_THRESHOLD)
        decision = apply_zone(detections, width, height, ROI, TARGET_CLASS)
        result_path = capture_path.with_name(capture_path.stem + "_zone.png")
        annotated = annotate_zone(capture_path, detections, ROI, decision)
        result_path = annotated
        timings["decode_zone_draw_s"] = time.perf_counter() - post_started
        switch_started = time.perf_counter()
        led_error = None
        try:
            set_alarm_led(decision["alarm"])
        except Exception as error:
            # Preserve successful detections even if base.bit/LED control is unavailable.
            led_error = str(error)
            print("LED update failed:", led_error)
            print("The screen result is valid. Set PHYSICAL_LED=False to continue without board LEDs.")
        timings["led_overlay_s"] = time.perf_counter() - switch_started
        timings["capture_to_result_s"] = time.perf_counter() - started
        record = dict(capture=memory_preview(capture_path), result=memory_preview(result_path),
                      threshold=THRESHOLD, roi=list(ROI), target=TARGET_CLASS,
                      counts=decision["counts"], target_in_zone=len(decision["inside"]),
                      alarm=decision["alarm"], physical_led=PHYSICAL_LED, led_error=led_error,
                      detections=detections, timings=timings)
        detection_history.append(record)
        print("Result {} — {}".format(len(detection_history), capture_path.name))
        show_pair(capture_path, result_path)
        print("Counts in this snapshot:", decision["counts"] or "none")
        print("{} inside zone: {} — {}".format(TARGET_CLASS, len(decision["inside"]),
              "ALARM" if decision["alarm"] else "CLEAR"))
        print("Capture to result + LED attempt: {:.3f} s; FPGA call: {:.3f} s; LED overlay: {:.3f} s".format(
            timings["capture_to_result_s"], timings["fpga_call_s"], timings["led_overlay_s"]))
        print("Includes camera warmup, overlay reload when needed, inference and image output; excludes notebook rendering.")
        return record
    except BaseException:
        classifier.deinit_accelerator()
        turn_led_off()
        raise
    finally:
        clear_intermediate_files()


def finish_session():
    for control in globals().get("ui_buttons", []):
        control.disabled = True
    try:
        turn_led_off()
    finally:
        release_model()
    print("Alarm LED off; FPGA runtime and Darknet released. Camera already closed.")


## 7. Try the alarm

Run the controls cell below. **Preview camera** gives a still preview. **Capture + detect** takes a new frame and appends its original/result pair. Wait for **Ready** between clicks.

Try these experiments:

1. Capture an empty target zone, then a person with the box center inside, then outside. Compare screen alarm and LED0.
2. Change `TARGET_CLASS = "bottle"`. Put a bottle inside/outside the zone and capture again.
3. Place two supported objects in the frame. Compare total per-class counts with the target count in the zone.
4. Move the ROI to the left half. Compare a box that overlaps the zone but whose center is outside it.

The alarm updates only after each button-triggered capture has finished processing. It is a teaching demo, not a continuous safety system. **Finish / release** turns off the LED and releases the model. Manual mode: call `capture_and_detect()` in a new cell.

In [ ]:
# Close old controls so a rerun cannot leave a second set of active buttons.
for control in globals().get("ui_buttons", []):
    control.disabled = True
if globals().get("ui_panel") is not None:
    ui_panel.close()

if widgets is not None:
    preview_button = widgets.Button(description="Preview camera", icon="camera")
    detect_button = widgets.Button(description="Capture + detect", icon="search", button_style="primary")
    finish_button = widgets.Button(description="Finish / release", icon="stop", button_style="warning")
    status = widgets.HTML("Ready. Aim the camera, optionally preview, then capture and detect.")
    history_output = widgets.Output()
    ui_buttons = [preview_button, detect_button, finish_button]
    ui_panel = widgets.VBox([widgets.HBox(ui_buttons), status, history_output])

    def perform(action):
        for control in ui_buttons:
            control.disabled = True
        status.value = "Working — wait for the current capture/inference to finish."
        try:
            with history_output:
                action()
            status.value = "Ready. Previous images remain below."
        except Exception:
            with history_output:
                traceback.print_exc()
            status.value = "Operation failed. See the message below; previous results are retained."
        finally:
            for control in ui_buttons:
                control.disabled = classifier is None

    def finish_clicked(_):
        for control in ui_buttons:
            control.disabled = True
        finish_session()
        status.value = "Session released. Rerun section 2 and this cell to start again."

    preview_button.on_click(lambda _: perform(preview_camera))
    detect_button.on_click(lambda _: perform(capture_and_detect))
    finish_button.on_click(finish_clicked)
    display(ui_panel)
else:
    ui_buttons = []
    ui_panel = None
    print("Manual mode: run preview_camera() or capture_and_detect() in a new cell.")


## 8. Redisplay completed results (optional)

This uses the in-memory previews without re-running inference or changing the physical LED.

In [ ]:
print("Completed detections:", len(detection_history))
for number, record in enumerate(detection_history, 1):
    print("Result", number)
    show_pair(record["capture"], record["result"])



## 9. Finish and clear memory

Run this cell or use **Finish / release**. It switches off the alarm LED, releases the native network/accelerator and clears large weights. No PNG/JSON results are saved. To restart, rerun section 2, configuration section 5 and controls section 7. Shut down this kernel before another FPGA notebook.

In [ ]:
finish_session()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Large weights cleared. No capture or result files were saved.")


Implementation references: [OpenCV VideoCapture](https://docs.opencv.org/3.4.20/d8/dfe/classcv_1_1VideoCapture.html), [QNN-MO-PYNQ](https://github.com/onigas/QNN-MO-PYNQ), [Darknet region layer](https://github.com/giuliogamba/darknet/blob/master/src/region_layer.c), and [PYNQ-Z2 base overlay](https://pynq.readthedocs.io/en/v2.5/pynq_overlays/pynqz2/pynqz2_base_overlay.html).

**Validation:** notebook syntax, camera/FPGA ownership, repeated initialization, in-memory history, callback cleanup and Python processing are checked off-board using simulated hardware/native interfaces. Actual USB camera, widgets, Darknet, FPGA timing/predictions and physical LED behavior must be verified on the PYNQ board.